# Run a slablet model and view it in Geode

Set the inputs in the next cell, then run all cells. The notebook:

1. builds a run config from the inputs (the mantle settings start from a base config);
2. tracks seafloor ages through the reconstruction, if needed (once per reconstruction; minutes);
3. runs the model from `START_MA` to `END_MA` (about 2 min per 1000 Myr once the reconstruction is cached);
4. draws the present-day field at three depths;
5. writes a Geode Archive of the run and opens the Geode viewer: in the last cell, and as a link for a
   new tab.

Paths are relative to the slablet repo (outputs go to `runs_real/<NAME>`). The viewer needs a Geode
checkout (`~/GIT/Geode`, or set `$GEODE`) with `npm install` done in `viewer/`; the first `show` builds
it once (about a minute).

In [ ]:
# ---- inputs ---------------------------------------------------------------------------------------
NAME = 'notebook_clennett2020'                 # outputs go to runs_real/<NAME>

# Which reconstruction (see slablet/gplates/models.py): gprm, plate_model_manager or files.
RECONSTRUCTION = {'source': 'gprm', 'model': 'Clennett', 'craton_files': []}
# e.g. Muller 2022 (optimised mantle frame) back to 1000 Ma; gprm lists no coastlines for it:
# RECONSTRUCTION = {'source': 'gprm', 'model': 'Muller2022', 'craton_files': ['Cratons/*.gpml'],
#                   'coastline_files': ['Coastlines/shapes_coastlines_Merdith_etal.gpmlz']}
ANCHOR_PLATE_ID = 0                            # must be a frame fixed to the mantle
START_MA, END_MA = 250, 0                      # ages in Ma (present day = 0); the run goes from START to END
DELTA_TIME = 1                                 # Myr per step
OUTPUT_EVERY_MYR = 10                          # one field (and one viewer frame) every this many Myr

BASE_CONFIG = 'configs/muller2022_refined.yaml'   # mantle settings to start from
MANTLE = {}                                    # overrides of the mantle settings, e.g. {'pile_flow_screen_deg': 45}
SEAFLOOR = True                                # seafloor ages at trenches and the lithosphere lid

VIEW = 'compare'                               # 'compare': run | REVEAL | SEMUCB-WM1;  'single': the run alone
DEPTH_KM = 1400                                # depth slice the viewer opens on
WITH_OPT1 = False                              # also show Muller 2022 OPT1 (only for a Muller 2022 run)
OPEN_TAB = False                               # also open the viewer in a browser tab

In [ ]:
import copy
import os
from pathlib import Path

# work from the slablet repo (relative paths in configs resolve against it)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO)
os.environ['SLABLET_WORK'] = str(REPO)

import yaml
import slablet
from slablet.gplates import models, run_mantle, seafloor_tracker

with open(BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)
for key in ('model_dir', 'rotation_files', 'topology_files'):      # the base's own reconstruction, if any
    cfg.pop(key, None)
cfg.update({'name': NAME, 'reconstruction': copy.deepcopy(RECONSTRUCTION), 'anchor_plate_id': ANCHOR_PLATE_ID,
            'start_ma': START_MA, 'end_ma': END_MA, 'delta_time': DELTA_TIME,
            'output_every_myr': OUTPUT_EVERY_MYR, 'output_dir': 'runs_real/' + NAME, 'seafloor': SEAFLOOR})
cfg['mantle'].update(MANTLE)
if not SEAFLOOR:
    cfg['mantle']['lithosphere_dVs'] = 0.0      # the lid needs the tracked seafloor ages

recon = models.load(cfg)
# the trench-sample and seafloor caches are named by the reconstruction, so every run of it shares them
cfg['name'] = 'nb-' + '-'.join(recon.label.lower().replace("'", '').replace(':', '').split()).replace('{', '').replace('}', '').replace(',', '')
print('reconstruction:', recon.label)
print('frame of anchor plate {}: {} ({})'.format(ANCHOR_PLATE_ID, recon.frame, recon.frame_note))
print('{} rotation files, {} topology files, coastlines: {}'.format(
    len(recon.rotation_files), len(recon.topology_files), 'yes' if recon.coastline_files else 'none'))
print('slablet', slablet.version_info())

In [ ]:
# seafloor ages and plate velocities through the reconstruction (cached; only the first run is slow)
if SEAFLOOR:
    p = slablet.MantleParams.from_config(cfg['mantle'])
    _, folder = seafloor_tracker.folder_for(cfg, p.slab_sample_deg)
    have = [os.path.exists(os.path.join(folder, 't{:04d}.npz'.format(int(a)))) for a in (START_MA, END_MA)]
    if all(have):
        print('seafloor cache found:', folder)
    else:
        seafloor_tracker.track(cfg, p.slab_sample_deg, start=START_MA, end=END_MA)

In [ ]:
summary = run_mantle.run(cfg)

A first look: the present-day field (dVs, %, blue = fast) at three depths, with the trenches of the end age.

In [ ]:
import numpy as np
import pygmt
import xarray as xr
from slablet.gplates.trench_samples import CachedTrenchSamples

ds = xr.open_dataset('runs_real/{}/mantle_temp_{:04.0f}Ma.nc'.format(NAME, END_MA))
trench = CachedTrenchSamples.for_config(cfg, slablet.MantleParams.from_config(cfg['mantle']).slab_sample_deg).samples(END_MA)
fig = pygmt.Figure()
pygmt.config(FONT_ANNOT_PRIMARY='7p', FONT_TITLE='9p', MAP_FRAME_TYPE='plain')
pygmt.makecpt(cmap='vik', series=[-3, 3], reverse=True)       # the viewer's default range
for i, z in enumerate([600, 1400, 2700]):
    if i:
        fig.shift_origin(xshift='7.5c')
    f = ds.dVs.interp(depth=z)
    f.gmt.registration, f.gmt.gtype = 1, 1
    fig.grdimage(f, region='g', projection='N150/7c', frame=['af', '+t{} km, {:.0f} Ma'.format(z, END_MA)])
    fig.plot(x=trench['lon'], y=trench['lat'], style='c0.03c', fill='black')
fig.colorbar(position='JBC+w8c/0.25c+o-7.5c/0.6c+h', frame=['x+ldVs (%)'])
fig.show()

In [ ]:
from slablet import geode

archive, model_id = geode.export_run(cfg, opt1=WITH_OPT1)
print(archive, model_id)

In [ ]:
geode.show(archive, model_id, mode=VIEW, depth_km=DEPTH_KM, open_tab=OPEN_TAB)